# Lab01 - sprint 1

## Random Variable 03 (r_def)

### Description

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Description (что такое центроид и r_def, смысл значений около 0 и 1, данные, почему величина случайная) -->

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# константы
SEED = 42
N_REP = 100        # повторов на размер выборки (состоятельность)
N_SAMPLE = 1000    # размер выборки с возвращением / без
N_MC = 500         # повторов Монте-Карло для sd среднего
N_GOF = 2000       # подвыборка для goodness_of_fit
B_GOF = 200        # n_mc_samples для goodness_of_fit
K_CHI2 = 20        # число равновероятных интервалов для chi2
EPS = 1e-6         # обрезка 0 и 1

rng = np.random.default_rng(SEED)

In [ ]:
df = pd.read_parquet("../wm811k_features.parquet")
x_all = df["r_def"]
x = x_all.dropna()
print("df.shape =", df.shape)
print("dieSize =", df["dieSize"].iloc[0], "| число лотов =", df["lotName"].nunique())
print(f"r_def определена: {len(x)} из {len(x_all)}, выпало {x_all.isna().mean():.4%}")
print("точных 0:", int((x == 0).sum()), "| точных 1:", int((x == 1).sum()))
print("n_defect == 1:", int((df["n_defect"] == 1).sum()), "| n_defect >= 2:", int((df["n_defect"] >= 2).sum()),
      "| min n_defect:", df["n_defect"].min())
x.describe()

#### Случайность: разброс внутри лотов

In [ ]:
by_lot = df.dropna(subset=["r_def"]).groupby("lotName")["r_def"].agg(["count", "min", "max", "mean", "std"])
by_lot[by_lot["count"] >= 10].head(10)

In [ ]:
# разложение дисперсии: внутри лотов и между лотами
g = df.dropna(subset=["r_def"]).groupby("lotName")["r_def"]
var_within = (g.var(ddof=1).fillna(0) * (g.size() - 1)).sum() / (len(x) - g.ngroups)
var_between = g.mean().var(ddof=1)
icc = var_between / (var_between + var_within)
pd.Series({"дисперсия всей выборки": x.var(ddof=1),
           "внутрилотовая компонента": var_within,
           "межлотовая компонента": var_between,
           "ICC": icc}).round(6)

In [ ]:
top = df["lotName"].value_counts().head(20).index
data = [df.loc[df["lotName"] == lot, "r_def"].dropna().values for lot in top]
fig, ax = plt.subplots(figsize=(11, 4))
ax.boxplot(data)
ax.set_xticks(range(1, len(top) + 1), top, rotation=90)
ax.set_xlabel("лот (lotName)")
ax.set_ylabel("r_def, доля радиуса")
ax.set_title("r_def внутри лотов (20 самых крупных лотов)")
plt.tight_layout()
plt.show()

### EDA & Distribution

#### Описательные статистики

In [ ]:
xv = x.to_numpy()
N = len(xv)
pd.Series({
    "n": N, "среднее": xv.mean(), "дисперсия (ddof=1)": xv.var(ddof=1), "ст. отклонение": xv.std(ddof=1),
    "медиана": np.median(xv), "q05": np.quantile(xv, 0.05), "q25": np.quantile(xv, 0.25),
    "q75": np.quantile(xv, 0.75), "q95": np.quantile(xv, 0.95), "min": xv.min(), "max": xv.max(),
    "асимметрия": stats.skew(xv), "эксцесс (избыточный)": stats.kurtosis(xv),
}).round(5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(xv, bins=60, density=True, edgecolor="k", linewidth=0.3)
ax.set_xlabel("r_def, доля радиуса")
ax.set_ylabel("плотность вероятности, 1/доля радиуса")
ax.set_title("Распределение r_def")
plt.show()

In [ ]:
# режимы по числу дефектов: случаев n_defect = 1 нет, группы по квинтилям n_defect
k_grp = pd.qcut(df.loc[x.index, "n_defect"], 5)
fig, ax = plt.subplots(figsize=(9, 4))
for grp, s in x.groupby(k_grp, observed=True):
    ax.hist(s, bins=50, density=True, histtype="step", lw=1.2, label=f"n_defect {grp}")
ax.set_xlabel("r_def, доля радиуса")
ax.set_ylabel("плотность вероятности, 1/доля радиуса")
ax.set_title("r_def по группам n_defect")
ax.legend()
plt.show()
x.groupby(k_grp, observed=True).agg(["count", "mean", "median", "std"]).round(4)

#### Подгонка распределений

In [ ]:
xc = np.clip(xv, EPS, 1 - EPS)
a_hat, b_hat, _, _ = stats.beta.fit(xc, floc=0, fscale=1)
_, s_ray = stats.rayleigh.fit(xv, floc=0)
c_wb, _, s_wb = stats.weibull_min.fit(xv, floc=0)

fits = {
    "бета": {"dist": stats.beta, "args": (a_hat, b_hat, 0, 1), "k": 2, "data": xc,
             "params": f"a={a_hat:.4f}, b={b_hat:.4f}"},
    "Рэлей": {"dist": stats.rayleigh, "args": (0, s_ray), "k": 1, "data": xv,
              "params": f"sigma={s_ray:.5f}"},
    "Вейбулл": {"dist": stats.weibull_min, "args": (c_wb, 0, s_wb), "k": 2, "data": xv,
                "params": f"c={c_wb:.4f}, lambda={s_wb:.5f}"},
}
for f in fits.values():
    f["ll"] = f["dist"].logpdf(f["data"], *f["args"]).sum()
    f["frozen"] = f["dist"](*f["args"])

fit_table = pd.DataFrame({name: {"параметры": f["params"], "число параметров": f["k"], "logL": f["ll"],
                                 "AIC": 2 * f["k"] - 2 * f["ll"],
                                 "среднее модели": f["frozen"].mean(), "1 - F(1)": f["frozen"].sf(1)}
                          for name, f in fits.items()}).T
fit_table["dAIC"] = fit_table["AIC"].astype(float) - fit_table["AIC"].astype(float).min()
fit_table

In [ ]:
# LR-тест: Рэлей (c = 2) вложен в Вейбулл
lr_rw = 2 * (fits["Вейбулл"]["ll"] - fits["Рэлей"]["ll"])
p_rw = stats.chi2.sf(lr_rw, df=1)
print(f"LR Рэлей vs Вейбулл = {lr_rw:.3f}, p = {p_rw:.3e}, c Вейбулла = {c_wb:.4f}")

In [ ]:
grid = np.linspace(0, xv.max() * 1.05, 400)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(xv, bins=60, density=True, color="lightgray", edgecolor="k", linewidth=0.3, label="данные")
for name, f in fits.items():
    ax.plot(grid, f["frozen"].pdf(grid), lw=1.5, label=name)
ax.set_xlabel("r_def, доля радиуса")
ax.set_ylabel("плотность вероятности, 1/доля радиуса")
ax.set_title("r_def: подогнанные плотности")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, f) in zip(axes, fits.items()):
    stats.probplot(f["data"], dist=f["dist"], sparams=f["args"], plot=ax)
    ax.set_title(f"QQ-plot r_def против: {name}")
    ax.set_xlabel("теоретические квантили, доля радиуса")
    ax.set_ylabel("выборочные квантили, доля радиуса")
plt.tight_layout()
plt.show()

#### Теоретический эталон: равномерные дефекты

In [ ]:
# масштаб Рэлея по группам n_defect против 1/(2*sqrt(k))
kd = df.loc[x.index, "n_defect"]
grp = pd.qcut(kd, 10)
ray_rows = []
for gname, s in x.groupby(grp, observed=True):
    k_med = kd[s.index].median()
    ray_rows.append({"группа n_defect": str(gname), "размер": len(s), "медиана k": k_med,
                     "sigma Рэлея (ММП)": np.sqrt((s**2).mean() / 2),
                     "теория 1/(2*sqrt(k))": 1 / (2 * np.sqrt(k_med))})
ray_tab = pd.DataFrame(ray_rows)
ray_tab["отношение"] = ray_tab["sigma Рэлея (ММП)"] / ray_tab["теория 1/(2*sqrt(k))"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ray_tab["медиана k"], ray_tab["sigma Рэлея (ММП)"], "o-", label="оценка по данным")
kk = np.linspace(kd.min(), kd.max(), 200)
ax.plot(kk, 1 / (2 * np.sqrt(kk)), "k--", label=r"$1/(2\sqrt{k})$")
ax.set_xlabel("n_defect (медиана в группе), чипов")
ax.set_ylabel("масштаб Рэлея sigma, доля радиуса")
ax.set_title("Масштаб Рэлея для r_def против теории равномерных дефектов")
ax.legend()
plt.show()
ray_tab.round(5)

In [ ]:
# нормированный центроид z = r_def * 2*sqrt(n_defect): при равномерных дефектах ~ Rayleigh(1)
z = (x * 2 * np.sqrt(kd)).to_numpy()
_, s_z = stats.rayleigh.fit(z, floc=0)
# эталонная смесь: для каждой пластины Rayleigh(1/(2*sqrt(k_i)))
ref = rng.rayleigh(1 / (2 * np.sqrt(kd.to_numpy())))
print(f"z: среднее = {z.mean():.4f} (Rayleigh(1): {np.sqrt(np.pi / 2):.4f}), sigma ММП = {s_z:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
gz = np.linspace(0, z.max(), 300)
axes[0].hist(z, bins=60, density=True, color="lightgray", edgecolor="k", linewidth=0.3, label="данные")
axes[0].plot(gz, stats.rayleigh.pdf(gz), "k--", label="Rayleigh(1)")
axes[0].plot(gz, stats.rayleigh.pdf(gz, scale=s_z), "C1", label=f"Rayleigh({s_z:.3f})")
axes[0].set_xlabel(r"$z = r_{def}\cdot 2\sqrt{n_{defect}}$, безразмерная")
axes[0].set_ylabel("плотность вероятности")
axes[0].set_title("Нормированный центроид")
axes[0].legend()
axes[1].hist(xv, bins=60, density=True, histtype="stepfilled", alpha=0.4, label="данные r_def")
axes[1].hist(ref, bins=60, density=True, histtype="step", lw=1.5, label="эталон: равномерные дефекты")
axes[1].set_xlabel("r_def, доля радиуса")
axes[1].set_ylabel("плотность вероятности, 1/доля радиуса")
axes[1].set_title("r_def против эталона равномерных дефектов")
axes[1].legend()
plt.tight_layout()
plt.show()

#### Математическая база

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Математическая база (PDF, параметры, моменты беты, Рэлея, Вейбулла; почему центроид ~ Рэлей; литература) -->

#### Нормализация

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Нормализация (зачем нужна, что изменилось) -->

In [ ]:
x_bc, lam_bc = stats.boxcox(xv)
print(f"Бокс-Кокс: lambda = {lam_bc:.4f}")
transforms = {
    "исходные": xv,
    "z-score": (xv - xv.mean()) / xv.std(ddof=1),
    "логит": np.log(xc / (1 - xc)),
    "Бокс-Кокс": x_bc,
}
pd.DataFrame({name: {"асимметрия": stats.skew(t), "эксцесс (избыточный)": stats.kurtosis(t)}
              for name, t in transforms.items()}).T.round(4)

In [ ]:
fig, axes = plt.subplots(2, len(transforms), figsize=(16, 7))
for j, (name, t) in enumerate(transforms.items()):
    axes[0, j].hist(t, bins=50, density=True, edgecolor="k", linewidth=0.3)
    axes[0, j].set_title(name)
    axes[0, j].set_xlabel("значение, усл. ед.")
    axes[0, j].set_ylabel("плотность")
    stats.probplot(t, dist="norm", plot=axes[1, j])
    axes[1, j].set_title(f"QQ-plot: {name}")
    axes[1, j].set_xlabel("квантили N(0,1)")
    axes[1, j].set_ylabel("выборочные квантили, усл. ед.")
plt.tight_layout()
plt.show()

### Independence & Estimations

#### Состоятельность оценок

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Состоятельность оценок -->

In [ ]:
# выборки с возвращением из всей совокупности, N_REP повторов на каждый размер
sizes = np.unique(np.logspace(1, np.log10(N), 15).astype(int))
true_vals = {"среднее": xv.mean(), "дисперсия": xv.var(ddof=1), "медиана": np.median(xv)}

rows = []
for n in sizes:
    est = np.array([[s.mean(), s.var(ddof=1), np.median(s)]
                    for s in (rng.choice(xv, n, replace=True) for _ in range(N_REP))])
    for j, name in enumerate(true_vals):
        q05, q50, q95 = np.quantile(est[:, j], [0.05, 0.5, 0.95])
        rows.append({"n": n, "оценка": name, "q05": q05, "медиана": q50, "q95": q95})
cons = pd.DataFrame(rows)
cons["ширина 5-95%"] = cons["q95"] - cons["q05"]
cons.pivot(index="n", columns="оценка", values="ширина 5-95%").round(6)

In [ ]:
units = {"среднее": "доля радиуса", "дисперсия": "доля радиуса²", "медиана": "доля радиуса"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, name in zip(axes, true_vals):
    cc = cons[cons["оценка"] == name]
    ax.fill_between(cc["n"], cc["q05"], cc["q95"], alpha=0.3, label="5-95% оценок")
    ax.semilogx(cc["n"], cc["медиана"], "o-", ms=3, label="медиана оценок")
    ax.axhline(true_vals[name], color="k", ls="--", label="генеральное значение")
    ax.set_xlabel("размер выборки n")
    ax.set_ylabel(f"{name}, {units[name]}")
    ax.set_title(f"Состоятельность: {name}")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
slopes = {}
for name in true_vals:
    cc = cons[(cons["оценка"] == name) & (cons["ширина 5-95%"] > 0)]
    slopes[name] = np.polyfit(np.log10(cc["n"]), np.log10(cc["ширина 5-95%"]), 1)[0]
    ax.loglog(cc["n"], cc["ширина 5-95%"], "o-", ms=3, label=f"{name}, наклон {slopes[name]:.3f}")
ax.set_xlabel("размер выборки n")
ax.set_ylabel("ширина полосы 5-95%, ед. оценки")
ax.set_title("Скорость сужения разброса оценок")
ax.legend()
plt.show()
pd.Series(slopes, name="наклон").round(4)

#### Выборки с возвращением и без

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Выборки с возвращением и без -->

In [ ]:
s_with = x.sample(N_SAMPLE, replace=True, random_state=SEED)
s_without = x.sample(N_SAMPLE, replace=False, random_state=SEED)

def summary(s):
    return pd.Series({"n": len(s), "среднее": s.mean(), "дисперсия": s.var(ddof=1),
                      "q05": s.quantile(0.05), "q25": s.quantile(0.25), "медиана": s.median(),
                      "q75": s.quantile(0.75), "q95": s.quantile(0.95)})

pd.DataFrame({"генеральная": summary(x), "с возвращением": summary(s_with),
              "без возвращения": summary(s_without)}).round(5)

In [ ]:
bins = np.linspace(0, xv.max(), 50)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(x, bins=bins, density=True, histtype="stepfilled", alpha=0.3, color="gray", label="генеральная")
ax.hist(s_with, bins=bins, density=True, histtype="step", lw=1.2, label="с возвращением")
ax.hist(s_without, bins=bins, density=True, histtype="step", lw=1.2, label="без возвращения")
ax.set_xlabel("r_def, доля радиуса")
ax.set_ylabel("плотность вероятности, 1/доля радиуса")
ax.set_title(f"Выборки n = {N_SAMPLE} и генеральная совокупность")
ax.legend()
plt.show()

In [ ]:
complement = x.drop(s_without.index)
for name, s, ref_s in [("с возвращением vs генеральная", s_with, x),
                       ("без возвращения vs дополнение", s_without, complement)]:
    r = stats.ks_2samp(s, ref_s)
    print(f"KS {name}: D = {r.statistic:.4f}, p = {r.pvalue:.4f}")

# квантили выборок против генеральной
qs = [0.05, 0.25, 0.5, 0.75, 0.95]
pd.DataFrame({"генеральная": x.quantile(qs), "с возвращением": s_with.quantile(qs),
              "без возвращения": s_without.quantile(qs)}).round(5)

In [ ]:
# Монте-Карло: sd выборочного среднего с возвращением и без против теории
sigma = xv.std(ddof=0)
n_grid = np.array([100, 500, 1000, 3000, 6000, 10000, 13000, 15000])
mc = []
for n in n_grid:
    m_with = [rng.choice(xv, n, replace=True).mean() for _ in range(N_MC)]
    m_without = [rng.choice(xv, n, replace=False).mean() for _ in range(N_MC)]
    mc.append({"n": n, "n/N": n / N, "sd с возвр.": np.std(m_with, ddof=1),
               "sd без возвр.": np.std(m_without, ddof=1),
               "теория sigma/sqrt(n)": sigma / np.sqrt(n),
               "теория с FPC": sigma / np.sqrt(n) * np.sqrt((N - n) / (N - 1))})
mc = pd.DataFrame(mc)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(mc["n"], mc["sd с возвр."], "o", label="эмпирически, с возвращением")
ax.loglog(mc["n"], mc["sd без возвр."], "s", label="эмпирически, без возвращения")
nn = np.linspace(n_grid.min(), n_grid.max(), 400)
ax.loglog(nn, sigma / np.sqrt(nn), "k-", lw=1, label=r"$\sigma/\sqrt{n}$")
ax.loglog(nn, sigma / np.sqrt(nn) * np.sqrt((N - nn) / (N - 1)), "k--", lw=1,
          label=r"$\sigma/\sqrt{n}\cdot\sqrt{(N-n)/(N-1)}$")
ax.set_xlabel("размер выборки n")
ax.set_ylabel("sd выборочного среднего, доля радиуса")
ax.set_title(f"Выборки с возвращением и без (N = {N})")
ax.legend()
plt.show()
mc.round(6)

#### Проверка гипотез

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Гипотезы (H0, H1, статистика, p-value, уровень значимости, вывод) -->

In [ ]:
# обычный KS с подогнанными параметрами и goodness_of_fit (Монте-Карло) на подвыборке
sub = rng.choice(xv, N_GOF, replace=False)
known = {"бета": {"loc": 0, "scale": 1}, "Рэлей": {"loc": 0}, "Вейбулл": {"loc": 0}}
gof_rows = []
for name, f in fits.items():
    data_full = f["data"]
    data_sub = np.clip(sub, EPS, 1 - EPS) if name == "бета" else sub
    ks_naive = stats.kstest(data_full, f["dist"].cdf, args=f["args"])
    row = {"KS D (вся выборка)": ks_naive.statistic, "KS p наивный": ks_naive.pvalue}
    for stat in ["ks", "ad"]:
        res = stats.goodness_of_fit(f["dist"], data_sub, known_params=known[name], statistic=stat,
                                    n_mc_samples=B_GOF, rng=rng)
        row[f"{stat.upper()} стат. (подвыборка)"] = res.statistic
        row[f"{stat.upper()} p Монте-Карло"] = res.pvalue
    gof_rows.append(pd.Series(row, name=name))
print(f"подвыборка N_GOF = {N_GOF}, n_mc_samples = {B_GOF}")
pd.DataFrame(gof_rows)

In [ ]:
# chi2 согласия по равновероятным интервалам, ddof = число параметров
chi_rows = []
for name, f in fits.items():
    edges = f["frozen"].ppf(np.linspace(0, 1, K_CHI2 + 1))
    O = np.histogram(f["data"], bins=edges)[0]
    E = np.full(K_CHI2, len(f["data"]) / K_CHI2)
    stat, p = stats.chisquare(O, E, ddof=f["k"])
    chi_rows.append({"модель": name, "chi2": stat, "df": K_CHI2 - 1 - f["k"], "p-value": p,
                     "min O": O.min(), "max O": O.max()})
chi_tab = pd.DataFrame(chi_rows).set_index("модель")
chi_tab

In [ ]:
# Манн-Уитни Training против Test: r_def и нормированный центроид z
lab = df.loc[x.index, "trianTestLabel"].to_numpy()
mw_rows = []
for name, vals in [("r_def", xv), ("z = r_def*2*sqrt(n_defect)", z),
                   ("n_defect", df.loc[x.index, "n_defect"].to_numpy())]:
    tr, te = vals[lab == "Training"], vals[lab == "Test"]
    r = stats.mannwhitneyu(tr, te, alternative="two-sided", method="asymptotic")
    mw_rows.append({"величина": name, "n Training": len(tr), "n Test": len(te),
                    "медиана Training": np.median(tr), "медиана Test": np.median(te),
                    "U": r.statistic, "p-value": r.pvalue, "U/(n1*n2)": r.statistic / (len(tr) * len(te))})
mw_tab = pd.DataFrame(mw_rows).set_index("величина")
mw_tab

#### Сводная таблица

In [ ]:
best = fit_table["AIC"].astype(float).idxmin()
summary_tab = pd.Series({
    "пластин (n)": N,
    "доля NaN": x_all.isna().mean(),
    "лотов": df["lotName"].nunique(),
    "среднее r_def": xv.mean(),
    "дисперсия r_def": xv.var(ddof=1),
    "медиана r_def": np.median(xv),
    "асимметрия": stats.skew(xv),
    "эксцесс": stats.kurtosis(xv),
    "внутрилотовая дисперсия": var_within,
    "межлотовая дисперсия": var_between,
    "ICC": icc,
    "бета a": a_hat, "бета b": b_hat,
    "Рэлей sigma": s_ray,
    "Вейбулл c": c_wb, "Вейбулл lambda": s_wb,
    "AIC бета": fit_table.loc["бета", "AIC"],
    "AIC Рэлей": fit_table.loc["Рэлей", "AIC"],
    "AIC Вейбулл": fit_table.loc["Вейбулл", "AIC"],
    "лучшая по AIC": best,
    "LR Рэлей vs Вейбулл": lr_rw, "p LR": p_rw,
    "теория sigma 1/(2*sqrt(медиана k))": 1 / (2 * np.sqrt(kd.median())),
    "z: sigma Рэлея": s_z,
    "Бокс-Кокс lambda": lam_bc,
    f"chi2 ({best})": chi_tab.loc[best, "chi2"],
    f"p chi2 ({best})": chi_tab.loc[best, "p-value"],
    "Манн-Уитни p (r_def)": mw_tab.loc["r_def", "p-value"],
    "Манн-Уитни U/(n1*n2) (r_def)": mw_tab.loc["r_def", "U/(n1*n2)"],
}, dtype=object)
summary_tab.map(lambda v: f"{v:.6g}" if isinstance(v, (float, np.floating)) else v).to_frame("значение")

#### Выводы

<!-- ТЕКСТ ПИШЕТ ДАРЬЯ: Выводы -->